# nstad_bench — Quickstart

**One dataset · one full config · end-to-end run with inline visualisation**

Dataset: **MIT-BIH Arrhythmia** DS1 → DS2 (inter-patient ECG)  
Task: binary — Normal vs Arrhythmia  
Domain shift: different patients

---

## 0. Prerequisites

```bash
pip install -e ".[dev]"
nstad-download mitbih          # ~100 MB, PhysioNet (no auth)
```

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # change to 'inline' when running in Jupyter
import matplotlib.pyplot as plt

# ── ensure repo root is on path ───────────────────────────────────────────────
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Python:', sys.version)
print('ROOT  :', ROOT)

## 1. Load MIT-BIH DS1 / DS2

The AAMI EC57 inter-patient split:  
- **DS1** (22 records) → source domain  
- **DS2** (22 records) → target domain  

Beat segmentation: 180-sample windows centred on each annotated R-peak,  
per-beat z-score normalisation.

In [ ]:
import wfdb
from functools import lru_cache

DATA_ROOT = Path.home() / '.nstad_bench' / 'data' / 'mitbih'

DS1 = ['101','106','108','109','112','114','115','116','118','119',
        '122','124','201','203','205','207','208','209','215','220','223','230']
DS2 = ['100','103','105','111','113','117','121','123','200','202','210',
        '212','213','214','219','221','222','228','231','232','233','234']

ARRHYTHMIA = set('AVLR/')
WIN = 90   # half-window → 180 samples total
MAX_PER_CLASS = 3000   # cap for faster demo

def _segment(records, root, max_per_class=MAX_PER_CLASS):
    X0, X1 = [], []
    for rec in records:
        try:
            sig, _ = wfdb.rdsamp(str(root / rec), channels=[0])
            ann    = wfdb.rdann(str(root / rec), 'atr')
        except Exception:
            continue
        s = sig[:, 0].astype(np.float32)
        s = (s - s.mean()) / (s.std() + 1e-8)
        for idx, sym in zip(ann.sample, ann.symbol):
            if sym not in set('NAVLRn/') or idx < WIN or idx + WIN >= len(s):
                continue
            beat = s[idx - WIN: idx + WIN]
            if sym in ARRHYTHMIA:
                X1.append(beat)
            elif sym in 'Nn':
                X0.append(beat)
    # Balance
    rng = np.random.default_rng(0)
    n = min(len(X0), len(X1), max_per_class)
    X0 = rng.choice(np.array(X0, np.float32), n, replace=False)
    X1 = rng.choice(np.array(X1, np.float32), n, replace=False)
    X  = np.concatenate([X0, X1])
    y  = np.array([0]*n + [1]*n, dtype=np.int64)
    order = rng.permutation(len(X))
    return X[order], y[order]

@lru_cache(maxsize=1)
def mitbih_loader():
    print('Loading DS1 …', end=' ')
    X_s, y_s = _segment(DS1, DATA_ROOT)
    print(f'{len(X_s)} beats')
    print('Loading DS2 …', end=' ')
    X_t, y_t = _segment(DS2, DATA_ROOT)
    print(f'{len(X_t)} beats')
    return X_s, y_s, X_t, y_t

X_s, y_s, X_t, y_t = mitbih_loader()
print(f'\nSource: {X_s.shape}  labels={np.bincount(y_s)}')
print(f'Target: {X_t.shape}  labels={np.bincount(y_t)}')

### Visualise sample beats

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 5), sharey=True)
names = ['Normal (source)', 'Arrhythmia (source)',
         'Normal (target)', 'Arrhythmia (target)']
data  = [(X_s[y_s==0], 'tab:blue'), (X_s[y_s==1], 'tab:red'),
         (X_t[y_t==0], 'tab:cyan'), (X_t[y_t==1], 'tab:orange')]
rng = np.random.default_rng(1)
t   = np.linspace(-0.25, 0.25, 180)
for col, (arr, color) in enumerate(data):
    idxs = rng.choice(len(arr), 2, replace=False)
    for row, i in enumerate(idxs):
        ax = axes[row, col]
        ax.plot(t, arr[i], color=color, lw=1.2)
        ax.axvline(0, color='k', lw=0.5, ls='--')
        ax.set_xlabel('Time (s)', fontsize=8)
        if row == 0:
            ax.set_title(names[col], fontsize=9)
        ax.tick_params(labelsize=7)
plt.suptitle('Sample ECG beats — MIT-BIH DS1 (source) and DS2 (target)', y=1.02)
plt.tight_layout()
plt.savefig('mitbih_sample_beats.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: mitbih_sample_beats.png')

## 2. Run the two-stage experiment pipeline

Config: `configs/mitbih.yaml`  
For this quickstart we use a **minimal subset** to complete in minutes:
- 1 representation (RawSignal)
- 1 model (InceptionTime1D, 5 epochs)
- 2 adaptation methods (SourceOnly + CoDATS)
- 1 seed, 3 HP trials

In [ ]:
import yaml, tempfile
from nstad_bench.experiments.runner import register_dataset, run_experiment

register_dataset('mitbih_ds1_ds2', mitbih_loader)

# Build quickstart config (trimmed for speed)
qs_cfg = {
    'experiment_name': 'mitbih_quickstart',
    'output_dir':      'results/',
    'n_bootstrap':     100,
    'screening':       {'top_k': 1, 'metric': 'delta_auc'},
    'random_search':   {'n_trials': 3, 'seeds': [0], 'base_seed': 42},
    'datasets':        ['mitbih_ds1_ds2'],
    'representations': ['RawSignal'],
    'models': {
        'InceptionTime1D': {
            'epochs': 5, 'lr': 1e-3, 'batch_size': 64,
            'nb_filters': 32, 'depth': 3,
        }
    },
    'adaptation_methods': {
        'SourceOnly': {},
        'CoDATS': {
            'n_epochs':      {'type': 'int',       'low': 5,    'high': 15},
            'lr':            {'type': 'log_float',  'low': 1e-5, 'high': 1e-3},
            'lr_disc':       {'type': 'log_float',  'low': 1e-4, 'high': 1e-2},
            'lambda_domain': {'type': 'log_float',  'low': 0.1,  'high': 5.0},
            'batch_size':    {'type': 'choice',     'choices': [32, 64]},
        },
    },
}

with tempfile.NamedTemporaryFile('w', suffix='.yaml', delete=False) as f:
    yaml.dump(qs_cfg, f)
    cfg_path = f.name

import logging
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s  %(levelname)-8s  %(message)s',
                    datefmt='%H:%M:%S')

df = run_experiment(cfg_path, config_root=ROOT / 'configs')
print(f'\nResult: {len(df)} rows')
df.groupby(['psi', 'metric_name'])['metric_value'].mean().unstack()

## 3. Summary tables

In [ ]:
from nstad_bench.analysis.summary_tables import method_summary, gain_by_dataset

print('=== Method summary ===')
display(method_summary(df, ci_columns=True).round(3))

print('\n=== Gain by dataset ===')
display(gain_by_dataset(df).round(3))

## 4. Visualisations

In [ ]:
import matplotlib
matplotlib.use('Agg')  # replace with 'inline' in Jupyter
import matplotlib.pyplot as plt
from nstad_bench.analysis.plots import plot_delta_auc_heatmap, plot_gain_barplot

fig1 = plot_delta_auc_heatmap(df, source_only=False,
                               title='ΔAUC — MIT-BIH quickstart')
fig1.savefig('mitbih_delta_auc.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: mitbih_delta_auc.png')

In [ ]:
fig2 = plot_gain_barplot(df, title='Gain(ψ) — MIT-BIH quickstart')
fig2.savefig('mitbih_gain.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: mitbih_gain.png')

## 5. ANOVA — which factor explains ΔAUC variance?

In [ ]:
from nstad_bench.analysis.anova import run_anova, effect_size_ranking

# Need ≥2 levels per factor; quickstart has 1 φ and 1 θ
# so we demonstrate ANOVA on ψ only.
try:
    result = run_anova(df, metric='delta_auc')
    print(result)
    print('\nEffect-size ranking:')
    display(effect_size_ranking(result))
except ValueError as e:
    print(f'Note: {e}')
    print('(Run with ≥2 representations AND ≥2 models for full ANOVA)')

## 6. One-command analysis pipeline

For the Parquet written by `run_experiment()`, the analysis pipeline
writes all tables and figures automatically:

In [ ]:
from nstad_bench.analysis.pipeline import analyze_experiment

parquet = Path('results') / 'mitbih_quickstart.parquet'
if parquet.exists():
    report = analyze_experiment(parquet, anova=False)
    print(report)
else:
    print(f'Parquet not found at {parquet} — run Section 2 first.')

## 7. CLI equivalent

Everything above can be reproduced from the command line:

```bash
# Download data
nstad-download mitbih

# Run full experiment (uses configs/mitbih.yaml)
python - <<'EOF'
from nstad_bench.experiments.runner import register_dataset, run_experiment
from my_loaders import mitbih_loader          # your loader module
register_dataset('mitbih_ds1_ds2', mitbih_loader)
run_experiment('configs/mitbih.yaml')
EOF

# Analyse results
nstad-analyze results/mitbih.parquet
```

---
Next: try **[CWRU](../configs/cwru.yaml)**, **[DeepBeat](../configs/deepbeat.yaml)**, or **[STEAD](../configs/stead.yaml)**.